# LeetCode 2075 — Decode the Slanted Ciphertext bằng Python

Link đề: https://leetcode.com/problems/decode-the-slanted-ciphertext/ · Độ khó: **Medium** · Chủ đề: String, Simulation

Skeleton LeetCode:

```python
class Solution:
    def decodeCiphertext(self, encodedText: str, rows: int) -> str:
        ...
```

## 1. Mục tiêu học

1. Hiểu mã hóa **chuyển vị chéo** (slanted transposition): văn bản được ghi theo các **đường chéo** của ma trận `rows × cols`, rồi đọc ra theo **hàng**.
2. Giải mã bằng **số học chỉ số**: ô `(r, c)` nằm ở vị trí `r·cols + c` trong `encodedText`; đường chéo bắt đầu ở `(0, c₀)` gồm các vị trí `c₀ + r·(cols + 1)`.
3. Suy ra `cols = len(encodedText) / rows` và giới hạn độ dài mỗi đường chéo `min(rows, cols − c₀)`.
4. Xử lý khoảng trắng: chỉ bỏ khoảng trắng **ở cuối** (`rstrip`) — khoảng trắng ở đầu/giữa là một phần văn bản.
5. Viết **bộ mã hóa** để sinh dữ liệu thử và kiểm chứng giải mã (encode ∘ decode = id).

## 2. Đề bài & ví dụ

Chuỗi `originalText` được mã hóa thành `encodedText` bằng một ma trận có số hàng cố định `rows`:
- `originalText` được ghi theo các **đường chéo** từ trên-trái xuống dưới-phải: đường chéo bắt đầu ở cột 0 trước, rồi cột 1, cột 2, … Ô trống được điền `' '`. Số cột được chọn sao cho cột phải cùng **không rỗng**.
- `encodedText` là toàn bộ ma trận đọc theo **hàng** (hàng 0 trái → phải, rồi hàng 1, …).

Ví dụ `originalText = "cipher"`, `rows = 3` ⇒ `encodedText = "ch   ie   pr"`.

Cho `encodedText` và `rows`, trả về `originalText` (không có khoảng trắng ở cuối; đáp án duy nhất).

**Ví dụ 1:** `"ch   ie   pr", 3` → `"cipher"`.

**Ví dụ 2:** `"iveo    eed   l te   olc", 4` → `"i love leetcode"`.

**Ví dụ 3:** `"coding", 1` → `"coding"`.

**Ràng buộc:** `0 ≤ len(encodedText) ≤ 10⁶`; chỉ chữ thường và `' '`; `1 ≤ rows ≤ 1000`.

In [1]:
import random
import string
import time
from typing import List

examples = [(("ch   ie   pr", 3), "cipher"), (("iveo    eed   l te   olc", 4), "i love leetcode"), (("coding", 1), "coding")]
for (enc, r), exp in examples:
    print(f"{enc!r}, rows={r} -> kỳ vọng {exp!r}")

'ch   ie   pr', rows=3 -> kỳ vọng 'cipher'
'iveo    eed   l te   olc', rows=4 -> kỳ vọng 'i love leetcode'
'coding', rows=1 -> kỳ vọng 'coding'


## 3. Kiến thức nền

### 3.1 Ma trận theo hàng (row-major)

Ma trận `rows × cols` đọc theo hàng thành chuỗi độ dài `rows·cols`; ô `(r, c)` ở vị trí `r·cols + c`.

In [2]:
enc, rows = examples[0][0]; cols = len(enc) // rows
for r in range(rows):
    print(f"hàng {r}: {list(enc[r * cols:(r + 1) * cols])}")

hàng 0: ['c', 'h', ' ', ' ']
hàng 1: [' ', 'i', 'e', ' ']
hàng 2: [' ', ' ', 'p', 'r']


### 3.2 Đường chéo bắt đầu ở cột `c₀`

Các ô `(0, c₀), (1, c₀+1), (2, c₀+2), …` — vị trí `r·cols + c₀ + r = c₀ + r·(cols + 1)`. Dừng khi `r = rows` hoặc `c₀ + r = cols`. Trong Python: `enc[c₀::cols + 1][:min(rows, cols − c₀)]`.

In [3]:
for c0 in range(cols):
    diag = enc[c0::cols + 1][:min(rows, cols - c0)]
    print(f"đường chéo từ cột {c0}: {diag!r}")

đường chéo từ cột 0: 'cip'
đường chéo từ cột 1: 'her'
đường chéo từ cột 2: '  '
đường chéo từ cột 3: ' '


### 3.3 `strip` và `rstrip`

`" a b ".strip()` bỏ cả khoảng trắng đầu; `rstrip()` chỉ bỏ cuối. Văn bản gốc có thể **bắt đầu** bằng khoảng trắng hoặc chứa nhiều khoảng trắng liên tiếp — chỉ phần **đuôi** là phần đệm của ma trận.

### 3.4 Mã hóa (để kiểm thử)

Theo hình minh họa của đề: văn bản được cắt thành các khúc dài `rows`; khúc thứ `j` nằm trọn trên đường chéo bắt đầu ở `(0, j)`. Số cột nhỏ nhất để mọi khúc nằm trọn là `max_j (j + len(khúc_j))` — khi đó cột phải cùng chứa ký tự cuối của khúc cuối nên không rỗng. Với `"cipher"`, `rows = 3`: khúc `"cip"`, `"her"` ⇒ `cols = max(0 + 3, 1 + 3) = 4`.

In [4]:
def encode(text, rows):
    # chia văn bản thành các khúc dài rows; khúc j nằm trên đường chéo bắt đầu ở (0, j)
    if not text:
        return ""
    pieces = [text[i:i + rows] for i in range(0, len(text), rows)]
    cols = max(j + len(pc) for j, pc in enumerate(pieces))        # nhỏ nhất để mọi khúc nằm trọn, cột cuối không rỗng
    grid = [[" "] * cols for _ in range(rows)]
    for j, pc in enumerate(pieces):
        for r, ch in enumerate(pc):
            grid[r][j + r] = ch
    return "".join("".join(row) for row in grid)

print(repr(encode("cipher", 3)), repr(encode("i love leetcode", 4)), repr(encode("coding", 1)))
assert encode("cipher", 3) == examples[0][0][0] and encode("i love leetcode", 4) == examples[1][0][0]

'ch   ie   pr' 'iveo    eed   l te   olc' 'coding'


## 4. Ý tưởng nền tảng / trực giác

Mã hóa = ghi theo đường chéo, đọc theo hàng. Giải mã làm ngược lại: dựng lại ma trận từ chuỗi (theo hàng), rồi **đọc theo đường chéo** cột 0, 1, …, nối lại và bỏ khoảng trắng cuối.

Tính tay **Ví dụ 1** (`rows = 3`, `cols = 12/3 = 4`):

| hàng | nội dung |
|---|---|
| 0 | `c h _ _` |
| 1 | `_ i e _` |
| 2 | `_ _ p r` |

Đường chéo: cột 0: `c, i, p`; cột 1: `h, e, r`; cột 2: `_, _`; cột 3: `_` ⇒ `"cipher   "` ⇒ `rstrip` ⇒ `"cipher"`.

## 5. Cách 1 — Brute Force / baseline: dựng ma trận 2D, đọc đường chéo

In [5]:
def decode_matrix(encodedText, rows):
    if not encodedText:
        return ""
    cols = len(encodedText) // rows
    grid = [list(encodedText[r * cols:(r + 1) * cols]) for r in range(rows)]
    out = []
    for c0 in range(cols):
        r, c = 0, c0
        while r < rows and c < cols:
            out.append(grid[r][c]); r += 1; c += 1
    return "".join(out).rstrip()

for (enc, r), exp in examples:
    got = decode_matrix(enc, r); print(repr(got), "kỳ vọng", repr(exp)); assert got == exp

'cipher' kỳ vọng 'cipher'
'i love leetcode' kỳ vọng 'i love leetcode'
'coding' kỳ vọng 'coding'


**Độ phức tạp:** `O(N)` thời gian với `N = len(encodedText)` (mỗi ô đọc một lần) nhưng `O(N)` bộ nhớ thêm cho ma trận. Đây đã là lời giải chấp nhận được; các cách sau gọn hơn.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (`cols`):** ma trận đầy đủ ⇒ `cols = N / rows` (chia hết).

**Insight 2 (chỉ số):** không cần ma trận: ô `(r, c₀ + r)` ở vị trí `c₀ + r(cols + 1)`.

**Insight 3 (dừng sớm):** khi đã đọc đủ phần có nghĩa, phần còn lại toàn khoảng trắng; `rstrip` ở cuối là đủ (văn bản gốc không có khoảng trắng cuối).

**Invariant:** sau khi đọc các đường chéo cột `0..c₀ − 1`, chuỗi kết quả là tiền tố tương ứng của "văn bản gốc + phần đệm".

### 6.1 Minh họa: thứ tự ghi trong ma trận của Ví dụ 2

In số thứ tự (vị trí trong văn bản gốc) của từng ô khi ghi theo đường chéo.

In [6]:
text, rows = "i love leetcode", 4
enc = encode(text, rows); cols = len(enc) // rows
order = [["  ."] * cols for _ in range(rows)]; pos = 0
for c0 in range(cols):
    r, c = 0, c0
    while r < rows and c < cols and pos < len(text):
        order[r][c] = f"{pos:>3}"; pos += 1; r += 1; c += 1
for row in order:
    print(" ".join(row))

  0   4   8  12   .   .
  .   1   5   9  13   .
  .   .   2   6  10  14
  .   .   .   3   7  11


## 7. Các cách tiếp theo

### 7.1 Cách 2 — Số học chỉ số (bản nộp)

In [7]:
class Solution:
    def decodeCiphertext(self, encodedText: str, rows: int) -> str:
        n = len(encodedText)
        if n == 0:
            return ""
        cols = n // rows
        out = []
        for c0 in range(cols):
            r = 0
            while r < rows and c0 + r < cols:
                out.append(encodedText[r * cols + c0 + r])
                r += 1
        return "".join(out).rstrip()

print([Solution().decodeCiphertext(*a) for a, _ in examples])

['cipher', 'i love leetcode', 'coding']


**Độ phức tạp Cách 2:** `O(N)` thời gian, `O(N)` cho chuỗi kết quả.

### 7.2 Cách 3 — Lát cắt bước nhảy `cols + 1`

In [8]:
def decode_slices(encodedText, rows):
    n = len(encodedText)
    if n == 0:
        return ""
    cols = n // rows
    return "".join(encodedText[c0::cols + 1][:min(rows, cols - c0)] for c0 in range(cols)).rstrip()

print([decode_slices(*a) for a, _ in examples])

['cipher', 'i love leetcode', 'coding']


Lát cắt `enc[c0::cols+1]` chạy bằng C nên rất nhanh; cắt bớt `[:min(rows, cols − c0)]` để không "tràn" sang đường chéo kế tiếp (vị trí `c₀ + r(cols+1)` với `c₀ + r ≥ cols` thuộc hàng sau nhưng cột nhỏ hơn).

### 7.3 Cách 4 — Dịch các hàng rồi `zip_longest` theo cột

Dịch hàng `r` sang trái `r` ô (`row[r:]`): khi đó đường chéo bắt đầu ở cột `c₀` trở thành **cột** `c₀` của các hàng đã dịch. `zip_longest` gom theo cột, tự động bỏ qua phần hàng đã "hết" (tương ứng điều kiện `c₀ + r < cols`).

In [9]:
from itertools import zip_longest
def decode_shift_zip(encodedText, rows):
    n = len(encodedText)
    if n == 0:
        return ""
    cols = n // rows
    shifted = [encodedText[r * cols + r:(r + 1) * cols] for r in range(rows)]      # hàng r bỏ r ký tự đầu
    return "".join(ch for col in zip_longest(*shifted, fillvalue="") for ch in col).rstrip()

print([decode_shift_zip(*a) for a, _ in examples])

['cipher', 'i love leetcode', 'coding']


Hàng `r` sau khi dịch có độ dài `cols − r` (có thể rỗng khi `r ≥ cols`); cột `c₀` của nó là ô `(r, c₀ + r)` — đúng phần tử thứ `r` của đường chéo `c₀`.

## 8. TRACE từng bước

Trace Cách 2 trên **Ví dụ 2** (`rows = 4`, `N = 24`, `cols = 6`):

In [10]:
def trace(enc, rows):
    n = len(enc); cols = n // rows; out = []
    print(f"N = {n}, cols = {cols}")
    for c0 in range(cols):
        seg = []
        r = 0
        while r < rows and c0 + r < cols:
            seg.append((r, c0 + r, enc[r * cols + c0 + r])); r += 1
        out += [ch for _, _, ch in seg]
        print(f"đường chéo cột {c0}: {[(r, c, ch) for r, c, ch in seg]}")
    res = "".join(out); print(f"nối: {res!r} -> rstrip: {res.rstrip()!r}")
    return res.rstrip()

assert trace(*examples[1][0]) == "i love leetcode"

N = 24, cols = 6
đường chéo cột 0: [(0, 0, 'i'), (1, 1, ' '), (2, 2, 'l'), (3, 3, 'o')]
đường chéo cột 1: [(0, 1, 'v'), (1, 2, 'e'), (2, 3, ' '), (3, 4, 'l')]
đường chéo cột 2: [(0, 2, 'e'), (1, 3, 'e'), (2, 4, 't'), (3, 5, 'c')]
đường chéo cột 3: [(0, 3, 'o'), (1, 4, 'd'), (2, 5, 'e')]
đường chéo cột 4: [(0, 4, ' '), (1, 5, ' ')]
đường chéo cột 5: [(0, 5, ' ')]
nối: 'i love leetcode   ' -> rstrip: 'i love leetcode'


| đường chéo | các ô (r, c) | ký tự |
|---|---|---|
| 0 | (0,0)(1,1)(2,2)(3,3) | `i`, ` `, `l`, `o` |
| 1 | (0,1)(1,2)(2,3)(3,4) | `v`, `e`, ` `, `l` |
| 2 | (0,2)(1,3)(2,4)(3,5) | `e`, `e`, `t`, `c` |
| 3 | (0,3)(1,4)(2,5) | `o`, `d`, `e` |
| 4, 5 | … | khoảng trắng đệm |

Nối lại: `"i love leetcode   "` ⇒ `rstrip` ⇒ `"i love leetcode"`.

## 9. Code nộp LeetCode

Bản chọn nộp: **Cách 2**.

In [11]:
class Solution:
    def decodeCiphertext(self, encodedText: str, rows: int) -> str:
        n = len(encodedText)
        if n == 0:
            return ""
        cols = n // rows
        out = []
        for c0 in range(cols):
            r = 0
            while r < rows and c0 + r < cols:
                out.append(encodedText[r * cols + c0 + r])
                r += 1
        return "".join(out).rstrip()

for (enc, r), exp in examples:
    got = Solution().decodeCiphertext(enc, r); print(repr(got), repr(exp)); assert got == exp

'cipher' 'cipher'
'i love leetcode' 'i love leetcode'
'coding' 'coding'


## 10. Vì sao thuật toán đúng

**Ma trận.** `encodedText` là ma trận `rows × cols` đọc theo hàng ⇒ `N = rows·cols` và ô `(r, c)` ở vị trí `r·cols + c`.

**Thứ tự ghi.** Văn bản gốc được ghi lần lượt vào các đường chéo bắt đầu ở cột `0, 1, …, cols − 1`, mỗi đường chéo từ trên xuống, dừng ở hàng cuối hoặc cột cuối. Đọc các ô theo **đúng** thứ tự đó (Cách 2) cho lại văn bản gốc, tiếp theo là các ô đệm (khoảng trắng) — vì văn bản chiếm một **tiền tố** của thứ tự ghi.

**Bỏ đệm.** Phần sau văn bản gốc toàn khoảng trắng; văn bản gốc không kết thúc bằng khoảng trắng ⇒ `rstrip` bỏ đúng phần đệm và không đụng tới văn bản. ∎

### 10.1 Kiểm chứng thực nghiệm: decode(encode(t)) = t

Sinh văn bản ngẫu nhiên (có khoảng trắng ở đầu và nhiều khoảng trắng liên tiếp, không có khoảng trắng cuối), mã hóa bằng `encode`, rồi giải mã bằng mọi cách.

In [12]:
def rand_text(rng, n):
    t = "".join(rng.choice("ab  ") for _ in range(n)).rstrip()
    return t if t else "a"

rng = random.Random(1010)
for _ in range(1500):
    t = rand_text(rng, rng.randint(1, 30)); rows = rng.randint(1, 6)
    enc = encode(t, rows)
    for f in (decode_matrix, Solution().decodeCiphertext, decode_slices, decode_shift_zip):
        assert f(enc, rows) == t, (f.__name__, t, rows, enc)
print("decode(encode(t)) = t cho 4 cách trên 1500 văn bản (có khoảng trắng đầu/giữa)")

decode(encode(t)) = t cho 4 cách trên 1500 văn bản (có khoảng trắng đầu/giữa)


## 11. Phân tích Time / Space Complexity

Gọi `N = len(encodedText) ≤ 10⁶`.

| Cách | Time | Space phụ |
|---|---|---|
| 1. Ma trận 2D | `O(N)` | `O(N)` |
| 2. Chỉ số | `O(N)` | `O(N)` (kết quả) |
| 3. Lát cắt | `O(N)` (C) | `O(N)` |
| 4. Dịch hàng + `zip_longest` | `O(N)` | `O(N)` |

In [13]:
rng = random.Random(2075)
t = rand_text(rng, 2 * 10 ** 5); rows = 400
# mã hóa nhanh bằng chỉ số để tạo dữ liệu lớn (ghi thẳng vào mảng phẳng)
def encode_fast(text, rows):
    if not text: return ""
    n = len(text); k = (n + rows - 1) // rows
    cols = max(k - 1 + (n - (k - 1) * rows), (k - 2 + rows) if k >= 2 else 0)
    grid = [" "] * (rows * cols)
    for pos, ch in enumerate(text):
        j, r = divmod(pos, rows)
        grid[r * cols + j + r] = ch
    return "".join(grid)

assert all(encode_fast(x, r) == encode(x, r) for x in ("cipher", "i love leetcode", "coding", "ab a") for r in (1, 2, 3, 4))
enc = encode_fast(t, rows); print("N =", len(enc))
for name, f in [("matrix", decode_matrix), ("index", Solution().decodeCiphertext), ("slices", decode_slices), ("shift_zip", decode_shift_zip)]:
    tt = time.perf_counter(); r = f(enc, rows); print(f"{name:<7}: {(time.perf_counter() - tt) * 1000:.0f} ms, đúng: {r == t}")

N = 359600
matrix : 22 ms, đúng: True
index  : 20 ms, đúng: True
slices : 1 ms, đúng: True


shift_zip: 8 ms, đúng: True


Ghi chú: `encode_fast` tính trực tiếp `cols = max((k−2) + rows, (k−1) + len(khúc cuối))` với `k = ⌈n/rows⌉` khúc, rồi ghi ký tự thứ `pos` (khúc `j = pos // rows`, hàng `r = pos % rows`) vào ô `(r, j + r)`. Kiểm tra trùng với `encode` trên các ví dụ nhỏ.

## 12. Bộ test

In [14]:
ALL = {"matrix": decode_matrix, "index": Solution().decodeCiphertext, "slices": decode_slices, "shift_zip": decode_shift_zip}
def run_all(enc, rows):
    res = {k: f(enc, rows) for k, f in ALL.items()}
    assert len(set(res.values())) == 1, (enc, rows, res)
    return res["index"]

for (enc, r), exp in examples:
    assert run_all(enc, r) == exp
# ví dụ tự dựng (đối chiếu lại đề gốc) — tạo bằng encode() rồi giải mã
edge_texts = [("", 3), ("a", 1), ("a", 5), (" a", 2), ("a  b", 3), ("  x", 4), ("ab", 2)]
for t, r in edge_texts:
    assert run_all(encode(t, r), r) == t, (t, r)
rng = random.Random(20750)
for _ in range(500):
    t = rand_text(rng, rng.randint(1, 50)); r = rng.randint(1, 8)
    assert run_all(encode_fast(t, r), r) == t
print("Ví dụ, 7 edge case (kể cả chuỗi rỗng, khoảng trắng đầu), 500 văn bản ngẫu nhiên: OK")

Ví dụ, 7 edge case (kể cả chuỗi rỗng, khoảng trắng đầu), 500 văn bản ngẫu nhiên: OK


### 12.1 Kiểm tra tính chất

- `rows = 1` ⇒ giải mã là `encodedText.rstrip()`.
- Độ dài kết quả `≤ N`; kết quả không kết thúc bằng khoảng trắng.
- Số ký tự khác khoảng trắng của kết quả bằng của `encodedText`.

In [15]:
rng = random.Random(121)
f = Solution().decodeCiphertext
for _ in range(1000):
    t = rand_text(rng, rng.randint(1, 40)); r = rng.randint(1, 7); enc = encode_fast(t, r); d = f(enc, r)
    assert len(d) <= len(enc) and not d.endswith(" ") and len(d.replace(" ", "")) == len(enc.replace(" ", ""))
    assert f(t, 1) == t.rstrip()
print("3 tính chất đúng trên 1000 văn bản")

3 tính chất đúng trên 1000 văn bản


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dùng `strip()` (bỏ cả khoảng trắng đầu)

In [16]:
def wrong_strip(enc, rows):
    return decode_matrix(enc, rows).strip() if enc else ""                     # SAI

t = "  hi there"                                                                # tự dựng (đối chiếu lại đề gốc)
e = encode(t, 3)
print("sai :", repr(wrong_strip(e, 3)), "| đúng:", repr(Solution().decodeCiphertext(e, 3)))

sai : 'hi there' | đúng: '  hi there'


**Vì sao sai:** văn bản gốc có thể bắt đầu bằng khoảng trắng; chỉ phần đệm **cuối** mới cần bỏ. **Sửa:** `rstrip()`.

### Lỗi 2 — Quên `rstrip` (trả cả phần đệm)

In [17]:
def wrong_no_rstrip(enc, rows):
    if not enc: return ""
    cols = len(enc) // rows
    return "".join(enc[c0::cols + 1][:min(rows, cols - c0)] for c0 in range(cols))   # SAI

for (enc, r), exp in examples:
    print("sai :", repr(wrong_no_rstrip(enc, r)), "| đúng:", repr(exp))

sai : 'cipher   ' | đúng: 'cipher'
sai : 'i love leetcode   ' | đúng: 'i love leetcode'
sai : 'coding' | đúng: 'coding'


**Vì sao sai:** các ô trống của ma trận (điền `' '`) nằm sau văn bản trong thứ tự đường chéo. **Sửa:** `rstrip()`.

### Lỗi 3 — Đọc theo **cột** thay vì đường chéo

In [18]:
def wrong_columns(enc, rows):
    if not enc: return ""
    cols = len(enc) // rows
    return "".join(enc[r * cols + c] for c in range(cols) for r in range(rows)).rstrip()   # SAI

for (enc, r), exp in examples:
    print("sai :", repr(wrong_columns(enc, r)), "| đúng:", repr(exp))

sai : 'c  hi  ep  r' | đúng: 'cipher'
sai : 'i   v   eel oe o dtl  ec' | đúng: 'i love leetcode'
sai : 'coding' | đúng: 'coding'


**Vì sao sai:** đây là giải mã của phép chuyển vị **thẳng** (cột), không phải chéo. **Sửa:** ô kế tiếp là `(r + 1, c + 1)`.

### Lỗi 4 — Không chặn đường chéo ở cột cuối (tràn sang hàng sau)

In [19]:
def wrong_no_bound(enc, rows):
    if not enc: return ""
    cols = len(enc) // rows
    return "".join(enc[c0::cols + 1][:rows] for c0 in range(cols)).rstrip()       # SAI: thiếu min(rows, cols - c0)

for (enc, r), exp in examples:
    print("sai :", repr(wrong_no_bound(enc, r)), "| đúng:", repr(exp))

sai : 'cipher' | đúng: 'cipher'
sai : 'i love leetcode' | đúng: 'i love leetcode'
sai : 'coding' | đúng: 'coding'


**Vì sao sai:** với `c₀ + r ≥ cols`, vị trí `c₀ + r(cols+1)` thuộc hàng **sau** ở cột đầu — không phải ô trên đường chéo. **Sửa:** giới hạn `min(rows, cols − c₀)`.

### Lỗi 5 — Chia cho 0 / lỗi với chuỗi rỗng

In [20]:
def wrong_empty(enc, rows):
    cols = len(enc) // rows
    try:
        return "".join(enc[c0::cols + 1][:min(rows, cols - c0)] for c0 in range(cols)).rstrip() + ("" if cols else enc[0])
    except IndexError as e:
        return f"IndexError: {e}"

print("encodedText rỗng:", repr(wrong_empty("", 3)), "| đúng:", repr(Solution().decodeCiphertext("", 3)))

encodedText rỗng: 'IndexError: string index out of range' | đúng: ''


**Vì sao sai:** `0 ≤ len(encodedText)` — chuỗi rỗng hợp lệ (văn bản gốc rỗng). Mọi truy cập chỉ số phải được bảo vệ. **Sửa:** trả `""` ngay khi `N = 0`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Mã hóa bằng ma trận ⇒ xác định công thức **chỉ số** của ô `(r, c)` trong chuỗi, tránh dựng ma trận.
- Đường chéo chính trong ma trận row-major: bước nhảy `cols + 1`.
- Khoảng trắng đệm ở cuối ⇒ `rstrip`, **không** `strip`.
- Viết bộ mã hóa để kiểm thử giải mã (round-trip).
- Liên quan: **6 Zigzag Conversion**, **498 Diagonal Traverse**.

## 15. Biến thể của bài

1. **6 Zigzag Conversion**: ghi theo đường zigzag, đọc theo hàng.
2. **498 Diagonal Traverse**: đọc ma trận theo đường chéo xen kẽ chiều.
3. **Đường chéo phụ** (trên-phải xuống dưới-trái): bước nhảy `cols − 1`.
4. **Mã hóa** (chiều ngược lại) với số cột tối thiểu — `encode_fast` ở mục 11.
5. **1424 Diagonal Traverse II**: ma trận răng cưa, gom theo `r + c`.

### 15.1 Code biến thể: 6 Zigzag Conversion (mã hóa và giải mã)

In [21]:
def zigzag(s, rows):
    if rows == 1: return s
    out = [[] for _ in range(rows)]; r, d = 0, 1
    for ch in s:
        out[r].append(ch)
        if r == 0: d = 1
        elif r == rows - 1: d = -1
        r += d
    return "".join("".join(x) for x in out)

def unzigzag(z, rows):
    n = len(z)
    if rows == 1: return z
    pattern = []; r, d = 0, 1
    for _ in range(n):
        pattern.append(r)
        if r == 0: d = 1
        elif r == rows - 1: d = -1
        r += d
    order = sorted(range(n), key=lambda i: (pattern[i], i))
    res = [""] * n
    for ch, i in zip(z, order): res[i] = ch
    return "".join(res)

rng = random.Random(6)
for _ in range(500):
    s = "".join(rng.choice("abcde") for _ in range(rng.randint(1, 20))); r = rng.randint(1, 6)
    assert unzigzag(zigzag(s, r), r) == s
print("Zigzag round-trip đúng;", zigzag("PAYPALISHIRING", 3), "(LeetCode 6: PAHNAPLSIIGYIR)")

Zigzag round-trip đúng; PAHNAPLSIIGYIR (LeetCode 6: PAHNAPLSIIGYIR)


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Ma trận 2D | Dựng lại, đọc chéo | `O(N)` | `O(N)` | Dễ hình dung |
| 2. Chỉ số | `r·cols + c₀ + r` | `O(N)` | `O(N)` | **Bản nộp** |
| 3. Lát cắt | `enc[c₀::cols+1]` | `O(N)` | `O(N)` | Nhanh nhất trong Python |
| 4. Dịch hàng + zip | Đường chéo thành cột | `O(N)` | `O(N)` | Kiểu Python gọn |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** (a) `("a", 1)`; (b) `("ab  ", 2)`; (c) `("a b c ", 2)` — kiểm tra bằng `encode` trước khi đoán; (d) `("", 5)`.

**Bài 2 (tính tay):** Mã hóa `"hello"` với `rows = 2` (tìm `cols`, vẽ ma trận) rồi giải mã lại.

**Bài 3 (điền TODO):** Hoàn thiện `diagonal(enc, rows, c0)` trả các ký tự của đường chéo bắt đầu ở cột `c0`.

**Bài 4 (tự viết lại):** Viết `encode_text(text, rows)` hiệu quả (`O(N)` sau khi tìm `cols`) và kiểm tra round-trip.

**Bài 5 (mini challenge):** Biến thể **đường chéo phụ**: văn bản ghi theo đường chéo bắt đầu ở `(0, c₀)` đi xuống **trái** `(r+1, c−1)`, các đường chéo theo thứ tự `c₀ = 0, 1, …` (bắt đầu từ cột 0 ở hàng 0, rồi cột 1, …, mở rộng thêm cột để chứa hết). Viết bộ mã hóa và giải mã tương ứng; kiểm tra round-trip.

In [22]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def diagonal(enc, rows, c0):
    cols = len(enc) // rows
    # TODO: trả "".join(enc[r * cols + c0 + r] for r in range(min(rows, cols - c0)))
    pass

In [23]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
def encode_text(text, rows):
    # TODO: tìm cols nhỏ nhất có sức chứa >= len(text); ghi theo đường chéo vào mảng phẳng
    pass

def anti_encode(text, rows):
    # TODO: như encode nhưng đường chéo đi (r + 1, c - 1)
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: `cols = N / rows`. Mức 2: đọc chéo + `rstrip`. Mức 3: (a) `"a"`; (b) ma trận `["ab", "  "]` ⇒ đường chéo `"a "`, `"b"` ⇒ `"a b"`; (c) tính bằng code; (d) `""`.

**Bài 2.** Mức 1: khúc `"he"`, `"ll"`, `"o"` ⇒ `cols = max(0+2, 1+2, 2+1) = 3`. Mức 2: ma trận `["hlo", " el"]`, `encodedText = "hlo el"`. Mức 3: giải mã lại phải ra `"hello"`.

**Bài 3.** Mức 1: độ dài đường chéo `min(rows, cols − c0)`. Mức 2: vị trí `r·cols + c0 + r`. Mức 3: nối chuỗi.

**Bài 4.** Mức 1: số khúc `k = ⌈n/rows⌉`. Mức 2: `cols = max_j (j + len(khúc_j))`. Mức 3: ghi ký tự `pos` vào vị trí `(pos % rows)·cols + pos // rows + pos % rows` của mảng phẳng.

**Bài 5.** Mức 1: đường chéo phụ từ `(0, c₀)` có độ dài `min(rows, c₀ + 1)`. Mức 2: chọn `cols` để đủ chứa. Mức 3: giải mã đọc lại theo đúng thứ tự đó.

## 19. Đáp án tham khảo cho bài tập

In [24]:
# Bài 1, 2, 3
f = Solution().decodeCiphertext
b1 = [f("a", 1), f("ab  ", 2), f("a b c ", 2), f("", 5)]
print("Bài 1:", b1); assert b1[0] == "a" and b1[1] == "a b" and b1[3] == ""
e = encode("hello", 2); print("Bài 2: encode('hello', 2) =", repr(e), "| ma trận:", [e[:len(e) // 2], e[len(e) // 2:]], "| giải mã:", f(e, 2)); assert f(e, 2) == "hello"
def diagonal(enc, rows, c0):
    cols = len(enc) // rows
    return "".join(enc[r * cols + c0 + r] for r in range(min(rows, cols - c0)))
assert [diagonal("ch   ie   pr", 3, c) for c in range(4)] == ["cip", "her", "  ", " "]
print("Bài 3: OK")

Bài 1: ['a', 'a b', 'ac  b', '']
Bài 2: encode('hello', 2) = 'hlo el' | ma trận: ['hlo', ' el'] | giải mã: hello
Bài 3: OK


In [25]:
# Bài 4, 5
def encode_text(text, rows):
    return encode_fast(text, rows)

def anti_encode(text, rows):
    n = len(text)
    if n == 0: return "", 0
    cols = 1
    while sum(min(rows, c0 + 1) for c0 in range(cols)) < n: cols += 1
    grid = [" "] * (rows * cols); pos = 0
    for c0 in range(cols):
        r, c = 0, c0
        while r < rows and c >= 0 and pos < n:
            grid[r * cols + c] = text[pos]; pos += 1; r += 1; c -= 1
    return "".join(grid), cols

def anti_decode(enc, rows):
    if not enc: return ""
    cols = len(enc) // rows; out = []
    for c0 in range(cols):
        r, c = 0, c0
        while r < rows and c >= 0:
            out.append(enc[r * cols + c]); r += 1; c -= 1
    return "".join(out).rstrip()

rng = random.Random(5)
for _ in range(500):
    t = rand_text(rng, rng.randint(1, 30)); r = rng.randint(1, 6)
    assert f(encode_text(t, r), r) == t
    e2, _ = anti_encode(t, r); assert anti_decode(e2, r) == t
print("Bài 4, 5: OK |", repr(anti_encode("cipher", 3)[0]), "->", anti_decode(anti_encode("cipher", 3)[0], 3))

Bài 4, 5: OK | 'cihpe r  ' -> cipher


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template:**

```python
if not enc: return ""
cols = len(enc) // rows
out = []
for c0 in range(cols):
    r = 0
    while r < rows and c0 + r < cols:
        out.append(enc[r * cols + c0 + r]); r += 1
return "".join(out).rstrip()
```

**Checklist:**
- [ ] `cols = N / rows`; xử lý `N = 0`?
- [ ] Ô kế tiếp trên đường chéo là `(r + 1, c + 1)` (không phải cột)?
- [ ] Chặn `c₀ + r < cols`?
- [ ] `rstrip` (không `strip`, không quên)?
- [ ] Đã test round-trip với văn bản có khoảng trắng đầu/giữa?